# Parts 8–9 — Ablation Study of the Proposed Pipeline

An ablation removes one component at a time while holding the corpus, query set, and remaining configuration fixed. This tests whether an observed gain depends on the intended component rather than merely on a more complicated pipeline.

**Protocol:** retrieval-stage leave-one-component-out on 120 development questions; reranker absent; locked test queries used = 0.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import html
import json
import platform
import random
import statistics

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(str(column).ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

def write_bar_svg(filename, values, title, *, maximum=None):
    values = list(values)
    width, left, right, row_height = 820, 245, 80, 34
    height = 76 + row_height * len(values)
    plot_width = width - left - right
    largest = maximum or max((float(value) for _, value in values), default=1.0) or 1.0
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width / 2}" y="27" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{html.escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values):
        y = 52 + index * row_height
        bar_width = plot_width * float(value) / largest
        elements.extend([
            f'<text x="{left - 10}" y="{y + 17}" text-anchor="end" font-family="Arial" font-size="13">{html.escape(str(label))}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="20" rx="3" fill="#1c5b58"/>',
            f'<text x="{min(left + bar_width + 7, width - 58):.2f}" y="{y + 16}" font-family="Arial" font-size="12">{float(value):.4g}</text>',
        ])
    elements.append('</svg>')
    target = FIGURES / filename
    target.write_text("\n".join(elements) + "\n", encoding="utf-8")
    print(f"Saved visualization: {target.relative_to(ROOT)}")
    return target

print(f"Project: {ROOT.name} | Python: {platform.python_version()} | fixed seed: {SEED}")


Project: h | Python: 3.12.13 | fixed seed: 20250816


## Components tested

- **Normalization:** reduces conservative Roman spelling variation.
- **Transliteration:** crosses the Latin/Urdu script boundary.
- **Expansion:** creates a retrieval-oriented query after controlled question-word removal.
- **BM25:** contributes exact Urdu lexical evidence.
- **Dense retrieval:** supplies multilingual semantic coverage.
- **Fusion:** combines heterogeneous ranked routes without comparing incompatible raw scores.

The stored ablation predates the romanized-title improvement; title-route impact is therefore reported separately as a before/after regression on the same questions rather than silently mixed into the older table.


In [2]:
report = load_json("reports/tables/retrieval_ablations.json")
assert report["queries"] == 120 and report["test_queries_used"] == 0
full = report["configurations"]["full_no_reranker"]
rows = []
for name, values in report["configurations"].items():
    rows.append({"configuration": name, "R@10": f'{values["recall_at_10"]:.4f}', "delta_R@10": f'{values["recall_at_10"] - full["recall_at_10"]:+.4f}', "MRR@10": f'{values["mrr_at_10"]:.4f}', "delta_MRR": f'{values["mrr_at_10"] - full["mrr_at_10"]:+.4f}', "mean_ms": f'{values["mean_latency_ms"]:.1f}'})
print_table(rows, ["configuration", "R@10", "delta_R@10", "MRR@10", "delta_MRR", "mean_ms"])


configuration      | R@10   | delta_R@10 | MRR@10 | delta_MRR | mean_ms
-------------------+--------+------------+--------+-----------+--------
full_no_reranker   | 0.1667 | +0.0000    | 0.0750 | +0.0000   | 425.2  
no_bm25            | 0.1417 | -0.0250    | 0.0529 | -0.0220   | 271.1  
no_dense           | 0.0833 | -0.0833    | 0.0488 | -0.0262   | 254.8  
no_expansion       | 0.1583 | -0.0083    | 0.0843 | +0.0094   | 354.4  
no_fusion          | 0.0583 | -0.1083    | 0.0232 | -0.0518   | 489.2  
no_normalization   | 0.1500 | -0.0167    | 0.0715 | -0.0035   | 402.9  
no_transliteration | 0.0667 | -0.1000    | 0.0253 | -0.0497   | 335.8  


## Recall visualization


In [3]:
values = [(name, result["recall_at_10"]) for name, result in report["configurations"].items()]
write_bar_svg("ablation_recall_at_10.svg", values, "Retrieval ablation Recall@10", maximum=0.20)


Saved visualization: reports\figures\ablation_recall_at_10.svg


![Ablation Recall at 10](../reports/figures/ablation_recall_at_10.svg)


## Component importance and trade-offs


In [4]:
deltas = []
for name, values in report["configurations"].items():
    if name == "full_no_reranker":
        continue
    deltas.append({"removed": name.removeprefix("no_"), "recall_loss": round(full["recall_at_10"] - values["recall_at_10"], 6), "mrr_change": round(values["mrr_at_10"] - full["mrr_at_10"], 6), "latency_saved_ms": round(full["mean_latency_ms"] - values["mean_latency_ms"], 3)})
print_table(sorted(deltas, key=lambda row: row["recall_loss"], reverse=True), ["removed", "recall_loss", "mrr_change", "latency_saved_ms"])
print("Largest Recall@10 dependency:", max(deltas, key=lambda row: row["recall_loss"])["removed"])


removed         | recall_loss | mrr_change | latency_saved_ms
----------------+-------------+------------+-----------------
fusion          | 0.108334    | -0.051773  | -63.941         
transliteration | 0.1         | -0.049686  | 89.442          
dense           | 0.083334    | -0.026227  | 170.487         
bm25            | 0.025       | -0.022037  | 154.161         
normalization   | 0.016667    | -0.003472  | 22.348          
expansion       | 0.008334    | 0.009352   | 70.817          
Largest Recall@10 dependency: fusion


Removing fusion causes the largest Recall@10 loss (0.1084), showing that no single route is sufficient. Transliteration and dense retrieval are also important. Removing expansion slightly lowers Recall@10 but raises MRR@10, so expansion is a mixed component rather than an automatic improvement. BM25 has a modest recall contribution but provides interpretable exact-term evidence.


## Separate title-route control


In [5]:
title = load_json("reports/tables/application_accuracy_regression.json")
title_rows = [
    {"system": "without romanized-title route", "R@1": title["before"]["recall_at_1"], "R@5": title["before"]["recall_at_5"], "R@10": title["before"]["recall_at_10"], "MRR@10": title["before"]["mrr_at_10"]},
    {"system": "with romanized-title route", "R@1": title["after"]["recall_at_1"], "R@5": title["after"]["recall_at_5"], "R@10": title["after"]["recall_at_10"], "MRR@10": title["after"]["mrr_at_10"]},
]
print_table(title_rows, ["system", "R@1", "R@5", "R@10", "MRR@10"])
print("Absolute title-route Recall@10 gain:", round(title["after"]["recall_at_10"] - title["before"]["recall_at_10"], 6))


system                        | R@1      | R@5      | R@10     | MRR@10  
------------------------------+----------+----------+----------+---------
without romanized-title route | 0.075    | 0.116667 | 0.191667 | 0.101362
with romanized-title route    | 0.391667 | 0.875    | 0.983333 | 0.583039
Absolute title-route Recall@10 gain: 0.791666


## Ablation conclusion

The original multi-view system depends most on fusion and transliteration, but its absolute recall remains low. The later romanized-title route produces the dominant measured gain because it targets the entity-matching failure directly. A future full ablation should remove the title route from the final frozen pipeline and repeat category-wise evaluation under independent review; the current before/after control is strong development evidence but not a locked-test result.
